In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

from langchain_ollama import ChatOllama
model = ChatOllama(model="gemma4:e2b")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user was asking a series of questions to establish details about a fictional location, "Lunapolis," its weather, population, and the status of its cheese miners\' union.\n\n## SUMMARY\nThe conversation established the following facts about the fictional location Lunapolis:\n*   **Capital:** Lunapolis.\n*   **Weather:** Skies are clear, with a high of 120C and a low of -100C.\n*   **Population:** 100,000 cheese miners live in Lunapolis.\n*   **Union Status:** The cheese miners\' union is expected to strike because they are unhappy with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='b58cb762-cba7-4a93-9708-7a0e29dae3ad'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user was asking a series of questions to establish details about a fictional location, "Lunapolis," its weather, population, and the status of its cheese miners' union.

## SUMMARY
The conversation established the following facts about the fictional location Lunapolis:
*   **Capital:** Lunapolis.
*   **Weather:** Skies are clear, with a high of 120C and a low of -100C.
*   **Population:** 100,000 cheese miners live in Lunapolis.
*   **Union Status:** The cheese miners' union is expected to strike because they are unhappy with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='0ad089c9-fee2-49e5-9324-bda2d584c1c3'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='7b499270-9cd3-4157-9324-8f9cf8a87458', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='a9d448ba-fe0c-433c-a325-d89bf0fc1d41'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='1056a673-36cb-4bd9-8a36-73862be74137', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='e258d312-beb1-486a-8e95-deb0ca9b97bc'),
              AIMessage(content='I understand that you are having trouble with your device.\n\n

In [8]:
print(response["messages"][-1].content)

I understand that you are having trouble with your device.

To help me figure out what is wrong, could you tell me:

1. **What kind of device is it?** (e.g., phone, laptop, TV, printer, etc.)
2. **What exactly is happening?** (e.g., Is there absolutely nothing that happens when you press the power button? Are there lights blinking? Is it making any noise?)
3. **Are there any lights or indicators on the device?**

Once I have this information, I can give you more specific troubleshooting steps.
